<a href="https://colab.research.google.com/github/Pumlisa97/mit805-airline-delays-pyspark/blob/main/notebooks/01_data_acquisition_and_profile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/MIT805_Airline_Delays/data")

for folder in ["raw", "working", "processed", "temp"]:
    path = DATA_ROOT / folder
    print(f"{folder:10} | exists: {path.exists()} | {path}")

raw        | exists: True | /content/drive/MyDrive/MIT805_Airline_Delays/data/raw
working    | exists: True | /content/drive/MyDrive/MIT805_Airline_Delays/data/working
processed  | exists: True | /content/drive/MyDrive/MIT805_Airline_Delays/data/processed
temp       | exists: True | /content/drive/MyDrive/MIT805_Airline_Delays/data/temp


In [6]:
RAW_DIR = DATA_ROOT / "raw"

for file in sorted(RAW_DIR.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 ** 2)
        print(f"{file.name} | {size_mb:.2f} MB")

On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip | 21.99 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.zip | 25.85 MB


In [8]:
import zipfile

for zip_path in sorted(RAW_DIR.glob("*.zip")):
    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_members = [
            item for item in archive.infolist()
            if item.filename.lower().endswith(".csv")
        ]
        csv_size_mb = sum(item.file_size for item in csv_members) / (1024 ** 2)

    print(f"{zip_path.name}")
    print(f"  ZIP size: {zip_path.stat().st_size / (1024 ** 2):.2f} MB")
    print(f"  CSV size: {csv_size_mb:.2f} MB")

On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip
  ZIP size: 21.99 MB
  CSV size: 201.83 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_1.zip
  ZIP size: 25.85 MB
  CSV size: 231.91 MB


In [10]:
import zipfile
from pathlib import Path

WORKING_DIR = DATA_ROOT / "working"
WORKING_DIR.mkdir(exist_ok=True)

for zip_path in sorted(RAW_DIR.glob("*.zip")):
    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_members = [
            item for item in archive.infolist()
            if item.filename.lower().endswith(".csv")
        ]

        for item in csv_members:
            target_path = WORKING_DIR / Path(item.filename).name

            if target_path.exists():
                print(f"Skipped (already exists): {target_path.name}")
            else:
                with archive.open(item) as source, open(target_path, "wb") as target:
                    target.write(source.read())
                print(f"Extracted: {target_path.name}")

Extracted: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
Extracted: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv


In [11]:
working_csvs = sorted(WORKING_DIR.glob("*.csv"))

for csv_path in working_csvs:
    size_mb = csv_path.stat().st_size / (1024 ** 2)
    print(f"{csv_path.name} | {size_mb:.2f} MB")

total_working_gb = sum(path.stat().st_size for path in working_csvs) / (1024 ** 3)
print(f"\nWorking CSV total: {total_working_gb:.3f} GiB")

On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv | 201.83 MB
On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv | 231.91 MB

Working CSV total: 0.424 GiB


In [12]:
import csv

for csv_path in working_csvs:
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as file:
        reader = csv.reader(file)
        header = next(reader)

    print(f"\n{csv_path.name}")
    print(f"Number of columns: {len(header)}")
    print(header)


On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
Number of columns: 110
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', '

In [13]:
import pandas as pd

for csv_path in working_csvs:
    sample = pd.read_csv(csv_path, nrows=5, low_memory=False)

    print(f"\n{'=' * 100}")
    print(csv_path.name)
    print(f"Rows sampled: {len(sample)} | Columns: {len(sample.columns)}")
    display(sample)


On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
Rows sampled: 5 | Columns: 110


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2015,1,1,1,4,2015-01-01,AA,19805,AA,N787AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,2,5,2015-01-02,AA,19805,AA,N795AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,3,6,2015-01-03,AA,19805,AA,N788AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,7,2015-01-04,AA,19805,AA,N791AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,5,1,2015-01-05,AA,19805,AA,N783AA,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv
Rows sampled: 5 | Columns: 110


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2025,1,1,1,3,2025-01-01,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1,1,2,4,2025-01-02,AA,19805,AA,N110AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025,1,1,3,5,2025-01-03,AA,19805,AA,N106NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025,1,1,4,6,2025-01-04,AA,19805,AA,N117AN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,1,1,5,7,2025-01-05,AA,19805,AA,N104NN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Initial schema and quality observations

The January 2015 and January 2025 BTS files each contain 110 columns and have
a compatible column order and naming structure. This supports a consistent
PySpark ingestion and transformation pipeline across the selected historical
period.

The final source-file header is blank. When loaded with Pandas, it appears as
`Unnamed: 109`; it functions as an extraneous row-number column and will be
removed before analysis.

The dataset contains many diversion-related fields (`Div1`–`Div5`). These are
expected to be sparse because diversions are relatively uncommon. They are not
required for the primary research question and will not be retained in the
core analytical dataset.

Cancelled and diverted flights may legitimately have missing actual-time,
elapsed-time, delay, and delay-cause values. Missingness must therefore be
interpreted in relation to flight operational status rather than treated
automatically as an error.

In [14]:
CORE_COLUMNS = [
    "Year",
    "Quarter",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Reporting_Airline",
    "DOT_ID_Reporting_Airline",
    "IATA_CODE_Reporting_Airline",
    "Flight_Number_Reporting_Airline",
    "Origin",
    "OriginCityName",
    "OriginState",
    "OriginStateName",
    "Dest",
    "DestCityName",
    "DestState",
    "DestStateName",
    "CRSDepTime",
    "DepTime",
    "DepDelay",
    "DepDelayMinutes",
    "DepDel15",
    "DepTimeBlk",
    "CRSArrTime",
    "ArrTime",
    "ArrDelay",
    "ArrDelayMinutes",
    "ArrDel15",
    "ArrTimeBlk",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Distance",
    "DistanceGroup",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

print(f"Core analytical columns: {len(CORE_COLUMNS)}")
print(CORE_COLUMNS)

Core analytical columns: 43
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateName', 'Dest', 'DestCityName', 'DestState', 'DestStateName', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepTimeBlk', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


In [15]:
for csv_path in working_csvs:
    sample = pd.read_csv(csv_path, nrows=1, low_memory=False)
    missing = [column for column in CORE_COLUMNS if column not in sample.columns]

    print(f"\n{csv_path.name}")
    if missing:
        print("Missing required columns:", missing)
    else:
        print("All 43 core columns are present.")


On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
All 43 core columns are present.

On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv
All 43 core columns are present.


Data scope :
January 2015 to December 2024

In [16]:
from pathlib import Path
from urllib.request import urlretrieve
from time import sleep

RAW_DIR = DATA_ROOT / "raw"
RAW_DIR.mkdir(exist_ok=True)

START_YEAR = 2015
END_YEAR = 2024

def bts_zip_filename(year: int, month: int) -> str:
    return (
        "On_Time_Reporting_Carrier_On_Time_Performance_"
        f"1987_present_{year}_{month}.zip"
    )

def bts_zip_url(year: int, month: int) -> str:
    return f"https://transtats.bts.gov/PREZIP/{bts_zip_filename(year, month)}"

download_plan = [
    (year, month, bts_zip_filename(year, month), bts_zip_url(year, month))
    for year in range(START_YEAR, END_YEAR + 1)
    for month in range(1, 13)
]

print(f"Planned files: {len(download_plan)}")
print(f"First file: {download_plan[0][3]}")
print(f"Last file:  {download_plan[-1][3]}")

Planned files: 120
First file: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip
Last file:  https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_12.zip


In [36]:
from urllib.error import HTTPError, URLError

YEAR_TO_DOWNLOAD = 2024

for year, month, filename, url in download_plan:
    if year != YEAR_TO_DOWNLOAD:
        continue

    destination = RAW_DIR / filename

    if destination.exists():
        print(f"Skipped existing: {filename}")
        continue

    try:
        print(f"Downloading {year}-{month:02d}: {filename}")
        urlretrieve(url, destination)
        size_mb = destination.stat().st_size / (1024 ** 2)
        print(f"Saved: {size_mb:.2f} MB")
        sleep(2)

    except (HTTPError, URLError, OSError) as error:
        print(f"FAILED for {year}-{month:02d}: {error}")

Saved: 26.30 MB
Saved: 23.92 MB
Saved: 27.96 MB
Saved: 27.22 MB
Saved: 29.70 MB
Saved: 29.78 MB
Saved: 32.83 MB
Saved: 29.43 MB
Saved: 27.12 MB
Saved: 28.06 MB
Saved: 26.85 MB
Saved: 28.75 MB


In [37]:
raw_zip_files = sorted(RAW_DIR.glob("*.zip"))

total_raw_zip_gib = sum(file.stat().st_size for file in raw_zip_files) / (1024 ** 3)

print(f"ZIP files in raw folder: {len(raw_zip_files)}")
print(f"Downloaded ZIP total: {total_raw_zip_gib:.3f} GiB")

for file in raw_zip_files:
    if "2015_" in file.name:
        print(f"{file.name} | {file.stat().st_size / (1024 ** 2):.2f} MB")

ZIP files in raw folder: 121
Downloaded ZIP total: 2.997 GiB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_1.zip | 21.99 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_10.zip | 22.77 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_11.zip | 22.51 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_12.zip | 23.33 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_2.zip | 20.25 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_3.zip | 23.66 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_4.zip | 22.53 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_5.zip | 23.20 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_6.zip | 24.57 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_7.zip | 24.99 MB
On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2015_8.zip | 24.54 MB
On_Time_Reporting_Carrier_On_Time_Performance

In [38]:
import re
import zipfile
from collections import defaultdict

def extract_year_month(filename: str):
    match = re.search(r"_(\d{4})_(\d{1,2})\.zip$", filename)
    return (int(match.group(1)), int(match.group(2))) if match else None

monthly_sizes = []

for zip_path in sorted(RAW_DIR.glob("*.zip")):
    parsed = extract_year_month(zip_path.name)

    if parsed is None:
        continue

    year, month = parsed

    # The defined project dataset is 2015–2024 only.
    if not (2015 <= year <= 2024):
        continue

    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_bytes = sum(
            item.file_size
            for item in archive.infolist()
            if item.filename.lower().endswith(".csv")
        )

    monthly_sizes.append(
        {
            "year": year,
            "month": month,
            "zip_name": zip_path.name,
            "zip_bytes": zip_path.stat().st_size,
            "csv_bytes": csv_bytes,
        }
    )

raw_csv_total_gib = sum(item["csv_bytes"] for item in monthly_sizes) / (1024 ** 3)
raw_zip_total_gib = sum(item["zip_bytes"] for item in monthly_sizes) / (1024 ** 3)

print(f"Historical monthly files counted: {len(monthly_sizes)}")
print(f"Compressed BTS ZIP collection: {raw_zip_total_gib:.3f} GiB")
print(f"Original uncompressed CSV collection: {raw_csv_total_gib:.3f} GiB")
print(f"Average monthly CSV size: {raw_csv_total_gib * 1024 / len(monthly_sizes):.2f} MiB")

Historical monthly files counted: 120
Compressed BTS ZIP collection: 2.971 GiB
Original uncompressed CSV collection: 26.495 GiB
Average monthly CSV size: 226.09 MiB


In [39]:
expected = {(year, month) for year in range(2015, 2025) for month in range(1, 13)}
found = {(item["year"], item["month"]) for item in monthly_sizes}

missing = sorted(expected - found)
duplicates = len(monthly_sizes) - len(found)

print(f"Expected files: {len(expected)}")
print(f"Found files: {len(found)}")
print(f"Missing year-months: {missing if missing else 'None'}")
print(f"Duplicate year-months: {duplicates}")

Expected files: 120
Found files: 120
Missing year-months: None
Duplicate year-months: 0


January 2020–December 2024

In [40]:
WORKING_START_YEAR = 2020
WORKING_END_YEAR = 2024

working_plan = [
    item
    for item in monthly_sizes
    if WORKING_START_YEAR <= item["year"] <= WORKING_END_YEAR
]

planned_working_gib = sum(item["csv_bytes"] for item in working_plan) / (1024 ** 3)
planned_working_gb = sum(item["csv_bytes"] for item in working_plan) / 1_000_000_000

print(f"Planned period: January {WORKING_START_YEAR} to December {WORKING_END_YEAR}")
print(f"Monthly files: {len(working_plan)}")
print(f"Planned extracted CSV size: {planned_working_gib:.3f} GiB")
print(f"Planned extracted CSV size: {planned_working_gb:.3f} GB")

Planned period: January 2020 to December 2024
Monthly files: 60
Planned extracted CSV size: 13.159 GiB
Planned extracted CSV size: 14.130 GB


In [41]:
import zipfile
from pathlib import Path

WORKING_DIR = DATA_ROOT / "working"
WORKING_DIR.mkdir(exist_ok=True)

for item in sorted(working_plan, key=lambda x: (x["year"], x["month"])):
    zip_path = RAW_DIR / item["zip_name"]

    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_members = [
            member
            for member in archive.infolist()
            if member.filename.lower().endswith(".csv")
        ]

        if len(csv_members) != 1:
            print(
                f"WARNING: expected one CSV in {zip_path.name}, "
                f"found {len(csv_members)}"
            )
            continue

        member = csv_members[0]
        target_path = WORKING_DIR / Path(member.filename).name

        if target_path.exists():
            print(f"Skipped existing: {target_path.name}")
            continue

        print(f"Extracting {item['year']}-{item['month']:02d}: {target_path.name}")

        with archive.open(member) as source, open(target_path, "wb") as target:
            while True:
                chunk = source.read(1024 * 1024 * 16)  # 16 MB at a time
                if not chunk:
                    break
                target.write(chunk)

Extracting 2020-01: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_1.csv
Extracting 2020-02: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_2.csv
Extracting 2020-03: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_3.csv
Extracting 2020-04: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_4.csv
Extracting 2020-05: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_5.csv
Extracting 2020-06: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_6.csv
Extracting 2020-07: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_7.csv
Extracting 2020-08: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_8.csv
Extracting 2020-09: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_9.csv
Extracting 2020-10: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2020_10.csv
Extracting 2020-11: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present

In [42]:
working_csvs = sorted(WORKING_DIR.glob("*.csv"))

working_total_bytes = sum(path.stat().st_size for path in working_csvs)
working_total_gib = working_total_bytes / (1024 ** 3)
working_total_gb = working_total_bytes / 1_000_000_000

print(f"Working CSV files: {len(working_csvs)}")
print(f"Working dataset size: {working_total_gib:.3f} GiB")
print(f"Working dataset size: {working_total_gb:.3f} GB")

working_year_months = set()
for path in working_csvs:
    parsed = extract_year_month(path.name.replace(".csv", ".zip"))
    if parsed:
        working_year_months.add(parsed)

expected_working = {
    (year, month)
    for year in range(2020, 2025)
    for month in range(1, 13)
}

missing_working = sorted(expected_working - working_year_months)

print(f"Expected working files: {len(expected_working)}")
print(f"Missing working periods: {missing_working if missing_working else 'None'}")

Working CSV files: 62
Working dataset size: 13.583 GiB
Working dataset size: 14.585 GB
Expected working files: 60
Missing working periods: None


In [43]:
import shutil

TEMP_DIR = DATA_ROOT / "temp"
TEMP_DIR.mkdir(exist_ok=True)

test_years = {"2015_1", "2025_1"}

for csv_path in WORKING_DIR.glob("*.csv"):
    if any(token in csv_path.name for token in test_years):
        destination = TEMP_DIR / csv_path.name

        if destination.exists():
            print(f"Already in temp: {destination.name}")
        else:
            shutil.move(str(csv_path), str(destination))
            print(f"Moved to temp: {csv_path.name}")

Moved to temp: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2015_1.csv
Moved to temp: On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv


In [44]:
working_csvs = sorted(WORKING_DIR.glob("*.csv"))

working_total_bytes = sum(path.stat().st_size for path in working_csvs)
working_total_gib = working_total_bytes / (1024 ** 3)
working_total_gb = working_total_bytes / 1_000_000_000

print(f"Working CSV files: {len(working_csvs)}")
print(f"Working dataset size: {working_total_gib:.3f} GiB")
print(f"Working dataset size: {working_total_gb:.3f} GB")

Working CSV files: 60
Working dataset size: 13.159 GiB
Working dataset size: 14.130 GB
